In [1]:
import os
import glob
import pandas as pd
from PIL import Image
from tqdm import tqdm
import shutil

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
# === 2️⃣ CHEMINS ===
root_dir = "/content/drive/MyDrive/tomato_classification_dataset_SOKOR"

In [6]:
# parcourir l'arborescence et compter
from collections import defaultdict

root = root_dir  # pointé sur le dossier racine où sont les classes
class_counts = defaultdict(int)
image_paths = []

for subdir, dirs, files in os.walk(root):
    for f in files:
        if f.lower().endswith(('.png','.jpg','.jpeg','.tif','.tiff', '.bmp')):
            full = os.path.join(subdir, f)
            image_paths.append(full)
            # la classe correspond au dossier parent immédiat (assumption standard)
            cls = os.path.basename(os.path.dirname(full))
            class_counts[cls] += 1

total_images = len(image_paths)
classes = sorted(class_counts.keys())

print("Total images :", total_images)
print("Nombre de classes :", len(classes))
print("Classes :", classes)
print("Images par classe :")
for c in classes:
    print(f"  {c} : {class_counts[c]}")

# obtenir taille/type d'une image exemple
from PIL import Image
if total_images>0:
    im = Image.open(image_paths[0])
    print("Exemple image :", image_paths[0])
    print("Taille (w,h) :", im.size, "Mode:", im.mode)
else:
    print("Aucune image trouvée.")


Total images : 18158
Nombre de classes : 10
Classes : ['0_TMBS', '1_TEB', '2_TLB', '3_TLM', '4_TSLS', '5_TSM', '6_TTS', '7_TYLCV', '8_TMV', '9_TH']
Images par classe :
  0_TMBS : 2127
  1_TEB : 1000
  2_TLB : 1907
  3_TLM : 952
  4_TSLS : 1771
  5_TSM : 1676
  6_TTS : 1404
  7_TYLCV : 5357
  8_TMV : 373
  9_TH : 1591
Exemple image : /content/drive/MyDrive/tomato_classification_dataset_SOKOR/9_TH/TMHE_image (1306).jpg
Taille (w,h) : (256, 256) Mode: RGB


## Redimensionner toutes les images en 224×224 pour VGG16

In [8]:
import os
from PIL import Image

# -------------------------------
# CONFIG
# -------------------------------
DATASET_DIR = "/content/drive/MyDrive/tomato_classification_dataset_SOKOR"  # dossier racine contenant les sous-dossiers des classes
TARGET_SIZE = (224, 224)  # requis par VGG16

def resize_all_images():
    # parcourir toutes les classes directement dans DATASET_DIR
    print(f"\n📁 Traitement du dataset : {DATASET_DIR}")

    for class_name in os.listdir(DATASET_DIR):
        class_dir = os.path.join(DATASET_DIR, class_name)

        if not os.path.isdir(class_dir):
            continue

        print(f"   🔄 Classe : {class_name}")

        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)

            if not img_name.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")):
                continue

            try:
                img = Image.open(img_path).convert("RGB")
                img = img.resize(TARGET_SIZE, Image.BILINEAR)
                img.save(img_path)
            except Exception as e:
                print(f"⚠️ Erreur image {img_path} → {e}")

    print("\n✅ Toutes les images ont été redimensionnées en 224×224 !")

resize_all_images()



📁 Traitement du dataset : /content/drive/MyDrive/tomato_classification_dataset_SOKOR
   🔄 Classe : 9_TH
   🔄 Classe : 7_TYLCV
   🔄 Classe : 8_TMV
   🔄 Classe : 6_TTS
   🔄 Classe : 4_TSLS
   🔄 Classe : 5_TSM
   🔄 Classe : 2_TLB
   🔄 Classe : 1_TEB
   🔄 Classe : 3_TLM
   🔄 Classe : 0_TMBS

✅ Toutes les images ont été redimensionnées en 224×224 !


## Augmentation

In [9]:
import os
import numpy as np
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# -----------------------------
# CONFIG
# -----------------------------
DATASET_DIR = "/content/drive/MyDrive/tomato_classification_dataset_SOKOR"
TARGET_SIZE = (224, 224)  # taille VGG16
SEED = 42

# Création de l'ImageDataGenerator pour augmentation + normalisation
augmenter = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

np.random.seed(SEED)

# -----------------------------
# 1. Calculer le nombre max d'images
# -----------------------------
class_counts = {}
for class_name in os.listdir(DATASET_DIR):
    class_dir = os.path.join(DATASET_DIR, class_name)
    if not os.path.isdir(class_dir):
        continue
    n_images = len([f for f in os.listdir(class_dir) if f.lower().endswith((".jpg",".jpeg",".png"))])
    class_counts[class_name] = n_images

max_images = max(class_counts.values())
print(f"✅ Nombre max d'images pour équilibrage : {max_images}")

# -----------------------------
# 2. Augmenter chaque classe jusqu'à max_images
# -----------------------------
for class_name, count in class_counts.items():
    class_dir = os.path.join(DATASET_DIR, class_name)
    if count >= max_images:
        print(f"Classe {class_name} déjà max ({count} images), pas d'augmentation.")
        continue

    print(f"🔄 Augmentation classe {class_name} : {count} -> {max_images}")
    images = [f for f in os.listdir(class_dir) if f.lower().endswith((".jpg",".jpeg",".png"))]

    # nombre d'images à générer
    n_to_generate = max_images - count

    for i in range(n_to_generate):
        # choisir image aléatoire
        img_name = np.random.choice(images)
        img_path = os.path.join(class_dir, img_name)

        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(TARGET_SIZE, Image.BILINEAR)
            img_array = np.expand_dims(np.array(img), 0)  # shape (1,h,w,c)

            # générer une image augmentée
            aug_iter = augmenter.flow(img_array, batch_size=1)
            aug_img = next(aug_iter)[0].astype(np.uint8)

            # sauvegarder
            new_name = f"{img_name.split('.')[0]}_aug_{i}.jpg"
            new_path = os.path.join(class_dir, new_name)
            Image.fromarray(aug_img).save(new_path)

        except Exception as e:
            print(f"⚠️ Erreur image {img_path} : {e}")

print("\n✅ Augmentation terminée ! Toutes les classes ont maintenant le même nombre d'images.")


✅ Nombre max d'images pour équilibrage : 5357
🔄 Augmentation classe 9_TH : 1591 -> 5357
Classe 7_TYLCV déjà max (5357 images), pas d'augmentation.
🔄 Augmentation classe 8_TMV : 373 -> 5357
🔄 Augmentation classe 6_TTS : 1404 -> 5357
🔄 Augmentation classe 4_TSLS : 1771 -> 5357
🔄 Augmentation classe 5_TSM : 1676 -> 5357
🔄 Augmentation classe 2_TLB : 1907 -> 5357
🔄 Augmentation classe 1_TEB : 1000 -> 5357
🔄 Augmentation classe 3_TLM : 952 -> 5357
🔄 Augmentation classe 0_TMBS : 2127 -> 5357

✅ Augmentation terminée ! Toutes les classes ont maintenant le même nombre d'images.


## test

In [10]:
# parcourir l'arborescence et compter
from collections import defaultdict

root = root_dir  # pointé sur le dossier racine où sont les classes
class_counts = defaultdict(int)
image_paths = []

for subdir, dirs, files in os.walk(root):
    for f in files:
        if f.lower().endswith(('.png','.jpg','.jpeg','.tif','.tiff', '.bmp')):
            full = os.path.join(subdir, f)
            image_paths.append(full)
            # la classe correspond au dossier parent immédiat (assumption standard)
            cls = os.path.basename(os.path.dirname(full))
            class_counts[cls] += 1

total_images = len(image_paths)
classes = sorted(class_counts.keys())

print("Total images :", total_images)
print("Nombre de classes :", len(classes))
print("Classes :", classes)
print("Images par classe :")
for c in classes:
    print(f"  {c} : {class_counts[c]}")

# obtenir taille/type d'une image exemple
from PIL import Image
if total_images>0:
    im = Image.open(image_paths[0])
    print("Exemple image :", image_paths[0])
    print("Taille (w,h) :", im.size, "Mode:", im.mode)
else:
    print("Aucune image trouvée.")


Total images : 53570
Nombre de classes : 10
Classes : ['0_TMBS', '1_TEB', '2_TLB', '3_TLM', '4_TSLS', '5_TSM', '6_TTS', '7_TYLCV', '8_TMV', '9_TH']
Images par classe :
  0_TMBS : 5357
  1_TEB : 5357
  2_TLB : 5357
  3_TLM : 5357
  4_TSLS : 5357
  5_TSM : 5357
  6_TTS : 5357
  7_TYLCV : 5357
  8_TMV : 5357
  9_TH : 5357
Exemple image : /content/drive/MyDrive/tomato_classification_dataset_SOKOR/9_TH/TMHE_image (1306).jpg
Taille (w,h) : (224, 224) Mode: RGB


## Normalisation

In [11]:
import os
import numpy as np
from PIL import Image
from tensorflow.keras.applications.vgg16 import preprocess_input

DATA_DIR = "/content/drive/MyDrive/tomato_classification_dataset_SOKOR"
OUTPUT_DIR = "/content/drive/MyDrive/tomato_normalized"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for class_name in os.listdir(DATA_DIR):
    class_in = os.path.join(DATA_DIR, class_name)
    class_out = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(class_out, exist_ok=True)

    for fname in os.listdir(class_in):
        if not fname.lower().endswith((".jpg", ".png", ".jpeg")):
            continue

        img = Image.open(os.path.join(class_in, fname)).convert("RGB")
        img = img.resize((224, 224))

        arr = np.array(img, dtype=np.float32)
        arr = np.expand_dims(arr, axis=0)
        arr = preprocess_input(arr)[0]

        # remettre dans un format affichable
        arr = arr.astype(np.float32)
        arr = arr - arr.min()
        arr = arr / arr.max() * 255

        img_norm = Image.fromarray(arr.astype(np.uint8))
        img_norm.save(os.path.join(class_out, fname))
